# CNN cost lab — ITMO Homework 1
One Kaggle GPU, FP32. Full 132-point grid; fixed train/validation split. Run all cells. Results are written to `/kaggle/working/hw1/results`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
try:
    import pynvml
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","nvidia-ml-py"])
root=Path("/kaggle/working/hw1"); root.mkdir(exist_ok=True); os.chdir(root)


In [ ]:
Path('models.py').write_text('"""Assignment architecture, no BatchNorm specified in the layer table."""\nimport torch\nfrom torch import nn\n\nclass SmallCNN(nn.Sequential):\n    def __init__(self):\n        layers=[]\n        for i,(ci,co,k,s) in enumerate([(3,32,7,2),(32,64,5,1),(64,128,3,2),(128,256,1,1),(256,256,3,2),(256,512,1,1)]):\n            layers += [nn.Conv2d(ci,co,k,stride=s,padding=k//2,bias=False), nn.ReLU(inplace=True)]\n            if i==0: layers += [nn.MaxPool2d(3,2,1)]\n        layers += [nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(512,256),nn.ReLU(inplace=True),nn.Linear(256,100)]\n        super().__init__(*layers)\n')


In [ ]:
Path('equations.py').write_text('"""FP32; MAC=2 FLOPs. Comparisons excluded. All functions broadcast."""\nimport numpy as np\nPARAMETERS = 1040324\nPARAMETER_BYTES = 4*PARAMETERS\n\ndef flops(image_size,batch):\n    s,b=np.broadcast_arrays(np.asarray(image_size,dtype=float),np.asarray(batch,dtype=float))\n    # Convolutions + GAP (n-1 adds and one division) + linear MACs + biases.\n    return b*(17714*s*s+313700)\n\ndef memory(image_size,batch):\n    s,b=np.broadcast_arrays(np.asarray(image_size,dtype=float),np.asarray(batch,dtype=float))\n    # Retained input + first conv activation + pooled output + int64 indices.\n    # Excludes opaque cuDNN workspace and allocator rounding; see derivations.\n    return PARAMETER_BYTES+68*b*s*s\n\ndef layer_costs(image_size,batch):\n    s,b=np.broadcast_arrays(np.asarray(image_size,dtype=float),np.asarray(batch,dtype=float))\n    z=np.zeros_like(s); F=[]; Q=[]\n    prev=3*s*s\n    for i,(ci,co,k,d) in enumerate([(3,32,7,2),(32,64,5,4),(64,128,3,8),(128,256,1,8),(256,256,3,16),(256,512,1,16)]):\n        out=co*(s/d)**2; w=ci*co*k*k\n        F.append(2*b*out*ci*k*k); Q.append(4*(b*(prev+out)+w))\n        F.append(z); Q.append(8*b*out) # separate in-place ReLU read/write\n        prev=out\n        if i==0:\n            out=32*(s/4)**2\n            F.append(z); Q.append(4*b*(prev+out)+8*b*out)\n            prev=out\n    F.append(b*prev); Q.append(4*b*(prev+512))\n    F.append(b*(2*512*256+256)); Q.append(4*(b*(512+256)+512*256+256))\n    F.append(z); Q.append(8*b*256)\n    F.append(b*(2*256*100+100)); Q.append(4*(b*(256+100)+256*100+100))\n    return np.stack(F,axis=-1),np.stack(Q,axis=-1)\n\ndef bytes_moved(image_size,batch):\n    return layer_costs(image_size,batch)[1].sum(axis=-1)\n\ndef latency(image_size,batch,theta):\n    f,q=layer_costs(image_size,batch)\n    return theta[\'launch_seconds\']*f.shape[-1]+np.maximum(f/theta[\'flops_per_second\'],q/theta[\'bytes_per_second\']).sum(axis=-1)\n\ndef energy(image_size,batch,theta_energy):\n    return (theta_energy[\'joules_per_flop\']*flops(image_size,batch)\n            +theta_energy[\'joules_per_byte\']*bytes_moved(image_size,batch)\n            +theta_energy[\'baseline_watts\']*latency(image_size,batch,theta_energy[\'latency\']))\n')


In [ ]:
Path('measure.py').write_text('"""Run on one Kaggle GPU: python measure.py; python calibrate.py."""\nimport csv,gc,json,platform,time,threading,subprocess\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom models import SmallCNN\nfrom equations import flops,memory\n\nFIELDS=[\'S\',\'B\',\'status\',\'latency\',\'memory\',\'energy\',\'is_validation\',\'flops\',\'memory_prediction\',\'energy_method\',\'energy_repeats\',\'energy_window_seconds\',\'power_samples\',\'error\']\n\ndef grid():\n    rng=np.random.default_rng(2026)\n    base_s=[32,64,128,224,256,384,512]; base_b=[1,2,4,8,16,32,64,128,256]\n    extra_s=sorted(rng.choice([s for s in range(32,513,16) if s not in base_s],4,replace=False).tolist())\n    extra_b=sorted(rng.choice([b for b in range(1,257) if b not in base_b],3,replace=False).tolist())\n    return [(s,b,s in extra_s or b in extra_b) for s in sorted(base_s+extra_s) for b in sorted(base_b+extra_b)]\n\nclass Power:\n    def __init__(self):\n        import pynvml as nv\n        nv.nvmlInit(); self.nv=nv\n        props=torch.cuda.get_device_properties(0)\n        try:self.h=nv.nvmlDeviceGetHandleByUUID(str(props.uuid))\n        except Exception:self.h=nv.nvmlDeviceGetHandleByIndex(0)\n        self.nv.nvmlDeviceGetPowerUsage(self.h)\n    def watts(self):return self.nv.nvmlDeviceGetPowerUsage(self.h)/1000\n    def counter(self):\n        try:return self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h)/1000\n        except Exception:return None\n    def run(self,fn,n):\n        samples=[]; stop=threading.Event()\n        def poll():\n            while not stop.is_set():\n                samples.append((time.perf_counter(),self.watts()));stop.wait(.02)\n        thread=threading.Thread(target=poll);thread.start()\n        e0=self.counter();t0=time.perf_counter();samples.append((t0,self.watts()))\n        try:\n            for _ in range(n):fn()\n            torch.cuda.synchronize()\n            t1=time.perf_counter();e1=self.counter();samples.append((t1,self.watts()))\n        finally:stop.set();thread.join()\n        if e0 is not None and e1 is not None and e1>e0:\n            return (e1-e0)/n,\'nvml_total_energy\',t1-t0,len(samples)\n        samples=sorted(samples)\n        times=np.array([x[0] for x in samples]);watts=np.array([x[1] for x in samples])\n        inner=(times>t0)&(times<t1)\n        tx=np.r_[t0,times[inner],t1];pw=np.interp(tx,times,watts)\n        return float(np.trapz(pw,tx))/n,\'nvml_power_integral\',t1-t0,len(samples)\n\ndef main():\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.manual_seed(2026);torch.set_num_threads(2)\n    assert torch.cuda.is_available(),\'GPU required: select Kaggle T4 accelerator\'\n    torch.cuda.set_device(0)\n    out=Path(\'results\');out.mkdir(exist_ok=True)\n    p=torch.cuda.get_device_properties(0)\n    try:power=Power();power_error=None\n    except Exception as e:power=None;power_error=str(e)\n    meta={\'gpu\':p.name,\'total_memory\':p.total_memory,\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'cudnn\':torch.backends.cudnn.version(),\'python\':platform.python_version(),\'seed\':2026,\'flags\':{\'benchmark\':False,\'cudnn_allow_tf32\':False,\'matmul_allow_tf32\':False},\'latency_protocol\':\'median of 9 individual host wall-clock forwards, synchronize before/after; 3 warmups\',\'energy_protocol\':\'whole GPU 0 only; target 1.2 sec sustained forwards (actual window saved in CSV); NVML counter, else sampled power integral; no idle subtraction\',\'power_error\':power_error,\'grid\':grid()}\n    (out/\'environment.json\').write_text(json.dumps(meta,indent=2))\n    with (out/\'measurements.csv\').open(\'w\',newline=\'\') as fp:\n        writer=csv.DictWriter(fp,fieldnames=FIELDS);writer.writeheader();fp.flush()\n        with torch.inference_mode():\n            model=SmallCNN().cuda().eval()\n            for index,(s,b,valid) in enumerate(grid()):\n                row=dict(S=s,B=b,is_validation=int(valid),flops=float(flops(s,b)),memory_prediction=float(memory(s,b)))\n                x=None;y=None\n                try:\n                    x=torch.randn(b,3,s,s,device=\'cuda\')\n                    for _ in range(3):y=model(x);del y\n                    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()\n                    y=model(x);torch.cuda.synchronize()\n                    row[\'memory\']=torch.cuda.max_memory_allocated();del y\n                    times=[]\n                    for _ in range(9):\n                        torch.cuda.synchronize();t=time.perf_counter();y=model(x);torch.cuda.synchronize();times.append(time.perf_counter()-t);del y\n                    row[\'latency\']=float(np.median(times))\n                    if power is not None:\n                        n=max(3,int(np.ceil(1.2/row[\'latency\'])))\n                        e,method,window,count=power.run(lambda:model(x),n)\n                        row.update(energy=e,energy_method=method,energy_repeats=n,energy_window_seconds=window,power_samples=count)\n                    else:row.update(energy_method=\'unavailable\',error=power_error)\n                    row[\'status\']=\'OK\'\n                except torch.cuda.OutOfMemoryError as e:\n                    row.update(status=\'OOM\',error=str(e).split(\'\\n\')[0]);y=None\n                finally:\n                    x=None;y=None;gc.collect();torch.cuda.empty_cache()\n                writer.writerow(row);fp.flush()\n                print(f\'{index+1}/132 S={s} B={b} {row["status"]} t={row.get("latency")} E={row.get("energy")}\',flush=True)\n    print(\'MEASUREMENTS COMPLETE\',flush=True)\nif __name__==\'__main__\':main()\n')


In [ ]:
Path('calibrate.py').write_text('"""Fit on base grid only. Random sizes/batches are held out before measuring."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import least_squares,nnls\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom equations import flops,memory,latency,energy,bytes_moved,layer_costs\n\ndef main():\n    out=Path(\'results\');fig=out/\'figures\';fig.mkdir(parents=True,exist_ok=True)\n    df=pd.read_csv(out/\'measurements.csv\');ok=df[df.status==\'OK\'].copy();train=ok[ok.is_validation==0]\n    s=train.S.to_numpy();b=train.B.to_numpy();t=train.latency.to_numpy()\n    def unpack(x):\n        v=np.exp(x);return dict(launch_seconds=v[0],flops_per_second=v[1],bytes_per_second=v[2])\n    fit=least_squares(lambda x:np.log(latency(s,b,unpack(x)))-np.log(t),np.log([1e-5,2e12,1e11]),bounds=(np.log([1e-9,1e8,1e7]),np.log([.1,1e15,1e14])))\n    theta=unpack(fit.x)\n    etrain=train[train.energy.notna() & (train.energy>0)]\n    te=None\n    if len(etrain)>=3:\n        se,be=etrain.S.to_numpy(),etrain.B.to_numpy()\n        X=np.column_stack([flops(se,be)/1e12,bytes_moved(se,be)/1e9,latency(se,be,theta)])\n        target=etrain.energy.to_numpy();coef,_=nnls(X/target[:,None],np.ones(len(target)))\n        te=dict(joules_per_flop=coef[0]/1e12,joules_per_byte=coef[1]/1e9,baseline_watts=coef[2],latency=theta)\n    (out/\'theta.json\').write_text(json.dumps({\'latency\':theta,\'energy\':te,\'training_rule\':\'base S x base B only; all random S or B held out\'},indent=2))\n    ss,bb=ok.S.to_numpy(),ok.B.to_numpy()\n    ok[\'latency_pred\']=latency(ss,bb,theta);ok[\'memory_pred\']=memory(ss,bb);ok[\'flops_pred\']=flops(ss,bb)\n    if te:ok[\'energy_pred\']=energy(ss,bb,te)\n    metrics={}\n    for name in [\'latency\',\'memory\']+([\'energy\'] if te else []):\n        metrics[name]={}\n        for split,v in [(\'train\',0),(\'validation\',1)]:\n            d=ok[(ok.is_validation==v)&ok[name].notna()];err=np.abs(d[name+\'_pred\']/d[name]-1)\n            metrics[name][split]={\'n\':len(d),\'median_absolute_relative_error\':float(np.median(err)),\'mean_absolute_relative_error\':float(np.mean(err)),\'p90_absolute_relative_error\':float(np.quantile(err,.9))}\n        factor,label={\'latency\':(1e3,\'Latency (ms)\'), \'memory\':(1/2**20,\'Peak allocated memory (MiB)\'), \'energy\':(1,\'Whole-GPU energy per forward (J)\')}[name]\n        plt.figure(figsize=(6,5))\n        for v,marker,lab in [(0,\'o\',\'Training\'),(1,\'x\',\'Held out\')]:\n            d=ok[(ok.is_validation==v)&ok[name].notna()]\n            plt.scatter(d[name]*factor,d[name+\'_pred\']*factor,label=lab,marker=marker,alpha=.7)\n        vals=np.r_[ok[name].dropna()*factor,ok[name+\'_pred\'].dropna()*factor];lo=max(vals.min()*.8,1e-10);hi=vals.max()*1.2\n        plt.plot([lo,hi],[lo,hi],\'k--\',label=\'Ideal\');plt.xscale(\'log\');plt.yscale(\'log\');plt.xlabel(\'Measured \'+label);plt.ylabel(\'Predicted \'+label);plt.legend();plt.tight_layout();plt.savefig(fig/(name+\'_parity.png\'),dpi=160);plt.close()\n        f,axes=plt.subplots(3,4,figsize=(15,10),sharex=True)\n        for ax,batch in zip(axes.flat,sorted(df.B.unique())):\n            d=ok[ok.B==batch].sort_values(\'S\');xs=np.linspace(32,512,160)\n            fn={\'latency\':lambda:latency(xs,batch,theta),\'memory\':lambda:memory(xs,batch),\'energy\':lambda:energy(xs,batch,te)}[name]\n            ax.plot(xs,fn()*factor,label=\'Prediction\')\n            for v,m,lab in [(0,\'o\',\'Training\'),(1,\'x\',\'Held out\')]:\n                part=d[d.is_validation==v];ax.scatter(part.S,part[name]*factor,marker=m,s=20,label=lab)\n            ax.set_title(f\'B={batch}\');ax.set_yscale(\'log\');ax.set_xlabel(\'S (pixels)\');ax.set_ylabel(label);ax.grid(alpha=.2)\n        axes.flat[0].legend(fontsize=7);f.tight_layout();f.savefig(fig/(name+\'_grid.png\'),dpi=140);plt.close(f)\n    env=json.loads((out/\'environment.json\').read_text())\n    df[\'predicted_oom\']=df.memory_prediction>env[\'total_memory\'];df.to_csv(out/\'oom_analysis.csv\',index=False)\n    ok.to_csv(out/\'predictions.csv\',index=False)\n    metrics[\'oom\']={\'observed\':int((df.status==\'OOM\').sum()),\'predicted_by_tensor_model\':int(df.predicted_oom.sum()),\'false_negatives\':int(((df.status==\'OOM\')&~df.predicted_oom).sum()),\'false_positives\':int(((df.status==\'OK\')&df.predicted_oom).sum())}\n    (out/\'metrics.json\').write_text(json.dumps(metrics,indent=2))\n    # FLOP accounting is checked with PyTorch profiler on 3 representative points.\n    if (out/\'flops_check.csv\').exists():\n        d=pd.read_csv(out/\'flops_check.csv\');plt.figure(figsize=(6,4));plt.plot(d.analytic_mac_flops,d.analytic_mac_flops,\'k--\',label=\'Analytic Conv/Linear\');plt.scatter(d.analytic_mac_flops,d.profiler_flops,label=\'PyTorch profiler\');plt.xlabel(\'Analytic Conv/Linear FLOPs\');plt.ylabel(\'Profiled Conv/Linear FLOPs\');plt.legend();plt.tight_layout();plt.savefig(fig/\'flops_check.png\',dpi=160);plt.close()\n    f,q=layer_costs(ss,bb);launch=np.full(len(ok),17*theta[\'launch_seconds\']);comp=(f/theta[\'flops_per_second\']).sum(-1);mem=(q/theta[\'bytes_per_second\']).sum(-1)\n    plt.figure(figsize=(7,5));sc=plt.scatter(flops(ss,bb)/bytes_moved(ss,bb),flops(ss,bb)/ok.latency/1e12,c=ok.S,cmap=\'viridis\',label=\'Measured\');plt.colorbar(sc,label=\'S (pixels)\');plt.scatter(flops(ss,bb)/bytes_moved(ss,bb),flops(ss,bb)/latency(ss,bb,theta)/1e12,c=\'tab:orange\',marker=\'x\',alpha=.6,label=\'Predicted\');plt.legend();plt.xscale(\'log\');plt.yscale(\'log\');plt.xlabel(\'Arithmetic intensity (FLOPs/byte, traffic model)\');plt.ylabel(\'Measured throughput (TFLOP/s)\');plt.tight_layout();plt.savefig(fig/\'regimes.png\',dpi=160);plt.close()\n    lines=[\'# Фактические результаты\',f\'GPU: {env["gpu"]}; PyTorch {env["torch"]}; CUDA {env["cuda"]}.\',f\'Всего {len(df)} точек, успешно {len(ok)}, OOM {metrics["oom"]["observed"]}.\',\'\', \'| Величина | Train MdAPE | Validation MdAPE | Validation P90 |\',\'|---|---:|---:|---:|\']\n    for name in [\'latency\',\'memory\']+([\'energy\'] if te else []):\n        m=metrics[name];lines.append(f\'| {name} | {m["train"]["median_absolute_relative_error"]:.1%} | {m["validation"]["median_absolute_relative_error"]:.1%} | {m["validation"]["p90_absolute_relative_error"]:.1%} |\')\n    lines+=[\'\', \'Параметры latency: `\'+json.dumps(theta)+\'`.\', \'Параметры energy: `\'+json.dumps(te)+\'`.\', \'\', \'## Обсуждение\', \'Модель latency суммирует roofline-оценку каждого оператора и постоянную стоимость запуска. Для малых тензоров Python, синхронизация и запуски занимают значимую долю; для больших растёт вклад вычислений. Разные cuDNN-алгоритмы и загрузка GPU вызывают отклонения, которые три глобальных параметра не описывают.\', \'Memory — аналитическая оценка живых тензоров с временными индексами MaxPool. Она не включает неизвестный workspace cuDNN и округление аллокатора. Поэтому она не гарантирует точный порог OOM; память устройства также расходуют CUDA-контекст и библиотеки. Расхождение нужно рассматривать как границу применимости, а не подгонять коэффициент памяти.\', \'Energy измерена для всей одной GPU в серии прямых проходов и поделена на число проходов. Это средняя энергия при устойчивой нагрузке, а не энергия изолированного запуска. Статическое потребление не вычиталось. Короткие проходы нельзя надёжно измерить одиночным чтением мощности. NVML-счётчик предпочтительнее; запасной способ — интеграл мощности с интервалом 20 мс. Вклад других процессов и дискретность датчика ограничивают точность.\', \'Коэффициенты энергии — эмпирическая декомпозиция, а не физические константы: FLOPs и bytes сильно коррелируют, поэтому нулевой коэффициент NNLS не доказывает отсутствие соответствующих затрат. Все параметры подбирались только по базовой сетке, случайные размеры/батчи не участвовали в fit.\', \'FLOPs учитывают Conv, Linear, bias и GAP. Сравнения ReLU/MaxPool считаются отдельно и не включаются в floating-point arithmetic; profiler проверяет только MAC-часть Conv/Linear.\']\n    (out/\'SUMMARY_RU.md\').write_text(\'\\n\'.join(line if line.startswith(\'|\') else \'\\n\'+line+\'\\n\' for line in lines))\n    print(json.dumps(metrics,indent=2))\nif __name__==\'__main__\':main()\n')


In [ ]:
import csv, torch
from models import SmallCNN
from equations import flops
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.allow_tf32=False
torch.backends.cuda.matmul.allow_tf32=False
Path("results").mkdir(exist_ok=True)
checks=[]
with torch.inference_mode():
    m=SmallCNN().cuda().eval()
    assert sum(p.numel() for p in m.parameters())==1040324
    for s,b in [(32,1),(128,2),(224,4)]:
        x=torch.randn(b,3,s,s,device="cuda")
        with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU],with_flops=True) as prof:
            y=m(x)
        measured=sum(e.flops for e in prof.key_averages())
        expected=b*(17712*s*s+313344)
        checks.append(dict(S=s,B=b,profiler_flops=measured,analytic_mac_flops=expected,total_analytic_flops=float(flops(s,b))))
        assert measured==expected,(measured,expected)
        del x,y
    del m
torch.cuda.empty_cache()
with open("results/flops_check.csv","w") as f:
    w=csv.DictWriter(f,fieldnames=list(checks[0]));w.writeheader();w.writerows(checks)
print(checks)

In [ ]:
from measure import main
main()

In [ ]:
from calibrate import main
main()

In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown(Path("results/SUMMARY_RU.md").read_text()))
for p in sorted(Path("results/figures").glob("*.png")):
    display(Image(filename=str(p)))
import shutil
shutil.make_archive("/kaggle/working/cnn-cost-lab-results","zip","/kaggle/working/hw1")